## DAY 1 — Clean data + Target-Anchored Contrastive MuRIL
Heads: Level1 (HATE/NON-HATE), Level3 (EXPLICIT/IMPLICIT), Level2 (Target, coarse).
Level4/5 dropped for now (too sparse) — flagged as future work in the writeup.
Novelty: supervised contrastive loss that pulls HATE examples with the same *target* together regardless of explicit/implicit wording, and pushes HATE vs NON-HATE apart — attacks the lexical-shortcut problem directly, inspired by DCL/AmpleHate/ImpCon (adapted, not copied, for a 5-level Tamil hierarchy).

In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, fbeta_score
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments, default_data_collator
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128
MIN_L4_COUNT = 25
MIN_L5_COUNT = 25
IMPLICIT_HATE_LOSS_MULT = 3.0
CONTRASTIVE_WEIGHT = 0.5     # lambda for the contrastive loss term
CONTRASTIVE_TEMP = 0.1

### Upload + load + clean

In [2]:
from google.colab import files
uploaded = files.upload()   # choose train.csv when the picker opens
csv_path = list(uploaded.keys())[0]

Saving train.csv to train.csv


In [3]:
df = pd.read_csv(csv_path)
print('raw shape:', df.shape)

df['Level1_Class'] = df['Level1_Class'].astype(str).str.upper().str.strip()
df = df[df['Level1_Class'].isin(['HATE', 'NON-HATE'])].copy()   # drops the 9 stray 'Nil' rows

df['Input_Text'] = df['Input_Text'].astype(str).str.strip()
df = df.drop_duplicates(subset=['Input_Text']).reset_index(drop=True)

for c in ['Level2_Target', 'Level3_Category', 'Level4_Subcategory', 'Level5_Bias']:
    df[c] = df[c].fillna('Nil').astype(str).str.strip()
df['Level3_Category'] = df['Level3_Category'].str.upper()

# fix the 'Men- no profane' labeling inconsistency -> fold into 'Hate against Men'
df['Level2_Target'] = df['Level2_Target'].replace({'Men- no profane': 'Hate against Men'})

print('cleaned shape:', df.shape)
print(df['Level1_Class'].value_counts())

raw shape: (4255, 7)
cleaned shape: (4242, 7)
Level1_Class
HATE        2230
NON-HATE    2012
Name: count, dtype: int64


In [4]:
# merge rare Level4/5 classes into 'Other' (Level4/5 heads not modeled this round, but keep columns clean for future work)
for c, min_n in [('Level4_Subcategory', MIN_L4_COUNT), ('Level5_Bias', MIN_L5_COUNT)]:
    vc = df[c].value_counts()
    rare = vc[vc < min_n].index
    df[c] = df[c].apply(lambda x: 'Other' if x in rare else x)

print(df['Level2_Target'].value_counts())

Level2_Target
Nil                    2040
Hate against Women     1025
Hate against Men        617
No Target- Profane      416
Hate against LGBTQ+     144
Name: count, dtype: int64


### Contrastive group label — the core novelty
HATE rows grouped by target (Women/Men/LGBTQ/NoTarget/Other), regardless of explicit vs implicit. NON-HATE is its own group. This is what the contrastive loss pulls together / pushes apart.

In [5]:
def contrastive_group(row):
    if row['Level1_Class'] == 'NON-HATE':
        return 'NONHATE'
    t = row['Level2_Target']
    if 'Women' in t: return 'HATE_WOMEN'
    if 'Men' in t and 'Women' not in t: return 'HATE_MEN'
    if 'LGBTQ' in t: return 'HATE_LGBTQ'
    if 'Profane' in t: return 'HATE_NOTARGET'
    return 'HATE_OTHER'

df['contrastive_group'] = df.apply(contrastive_group, axis=1)
print(df['contrastive_group'].value_counts())

contrastive_group
NONHATE          2012
HATE_WOMEN       1019
HATE_MEN          610
HATE_NOTARGET     412
HATE_LGBTQ        143
HATE_OTHER         46
Name: count, dtype: int64


In [6]:
LEVELS = ['Level1_Class', 'Level2_Target', 'Level3_Category']
encoders = {c: LabelEncoder().fit(df[c]) for c in LEVELS}
encoders['contrastive_group'] = LabelEncoder().fit(df['contrastive_group'])
for c in LEVELS:
    df[c + '_enc'] = encoders[c].transform(df[c])
df['group_enc'] = encoders['contrastive_group'].transform(df['contrastive_group'])

num_classes = {c: len(encoders[c].classes_) for c in LEVELS}
HATE_IDX = list(encoders['Level1_Class'].classes_).index('HATE')
IMPLICIT_IDX = list(encoders['Level3_Category'].classes_).index('IMPLICIT')
print(num_classes)

{'Level1_Class': 2, 'Level2_Target': 5, 'Level3_Category': 3}


In [7]:
df['strat_key'] = df['Level1_Class'] + '_' + df['Level3_Category']
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['strat_key'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['strat_key'], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

train_df.to_csv('train_clean.csv', index=False)
val_df.to_csv('val_clean.csv', index=False)
test_df.to_csv('test_clean.csv', index=False)

2969 636 637


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def build_ds(d):
    ds = Dataset.from_pandas(d.reset_index(drop=True))
    def enc(batch):
        tok = tokenizer(batch['Input_Text'], padding='max_length', truncation=True, max_length=MAX_LEN)
        tok['label1'] = batch['Level1_Class_enc']
        tok['label2'] = batch['Level2_Target_enc']
        tok['label3'] = batch['Level3_Category_enc']
        tok['group'] = batch['group_enc']
        tok['mask'] = [1.0 if l == HATE_IDX else 0.0 for l in batch['Level1_Class_enc']]
        return tok
    return ds.map(enc, batched=True, remove_columns=ds.column_names)

train_ds, val_ds, test_ds = build_ds(train_df), build_ds(val_df), build_ds(test_df)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Map:   0%|          | 0/2969 [00:00<?, ? examples/s]

Map:   0%|          | 0/636 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

### Model: shared MuRIL backbone, classification heads + a projection head for supervised contrastive loss

In [9]:
def supcon_loss(proj, group_labels, temp=0.1):
    """Supervised contrastive loss (Khosla et al.), in-batch positives = same contrastive_group."""
    proj = F.normalize(proj, dim=1)
    sim = torch.matmul(proj, proj.T) / temp
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()  # numerical stability

    labels = group_labels.unsqueeze(0) == group_labels.unsqueeze(1)
    self_mask = torch.eye(labels.shape[0], device=labels.device).bool()
    pos_mask = labels & ~self_mask

    exp_sim = torch.exp(sim) * (~self_mask)
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

    pos_count = pos_mask.sum(dim=1)
    valid = pos_count > 0
    if valid.sum() == 0:
        return torch.tensor(0.0, device=proj.device)
    mean_log_prob_pos = (log_prob * pos_mask).sum(dim=1)[valid] / pos_count[valid]
    return -mean_log_prob_pos.mean()


class ContrastiveMuril(nn.Module):
    def __init__(self, model_name, num_classes, implicit_mult, con_weight, con_temp):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({
            'label1': nn.Linear(hidden, num_classes['Level1_Class']),
            'label2': nn.Linear(hidden, num_classes['Level2_Target']),
            'label3': nn.Linear(hidden, num_classes['Level3_Category']),
        })
        self.proj_head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Linear(256, 128))
        self.implicit_mult = implicit_mult
        self.con_weight = con_weight
        self.con_temp = con_temp
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, input_ids, attention_mask, label1=None, label2=None, label3=None, group=None, mask=None, **kw):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        logits = {k: h(cls) for k, h in self.heads.items()}

        loss = None
        if label1 is not None:
            per_sample_l1 = self.ce(logits['label1'], label1)
            is_implicit_hate = (label1 == HATE_IDX) & (label3 == IMPLICIT_IDX)
            w = torch.where(is_implicit_hate, torch.full_like(per_sample_l1, self.implicit_mult), torch.ones_like(per_sample_l1))
            l1 = (per_sample_l1 * w).mean()
            l3 = self.ce(logits['label3'], label3).mean()

            m = mask if mask is not None else torch.ones_like(label1, dtype=torch.float)
            denom = m.sum().clamp(min=1.0)
            l2 = (self.ce(logits['label2'], label2) * m).sum() / denom

            proj = self.proj_head(cls)
            l_con = supcon_loss(proj, group, temp=self.con_temp)

            loss = l1 + l3 + 0.5*l2 + self.con_weight * l_con

        return {'loss': loss, 'logits1': logits['label1'], 'logits2': logits['label2'], 'logits3': logits['label3']}


class ContrastiveTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        outputs = model(**inputs)
        loss = outputs['loss']
        return (loss, outputs) if return_outputs else loss

In [10]:
model = ContrastiveMuril(MODEL_NAME, num_classes, IMPLICIT_HATE_LOSS_MULT, CONTRASTIVE_WEIGHT, CONTRASTIVE_TEMP)

args = TrainingArguments(
    output_dir='./out_con',
    num_train_epochs=6,
    per_device_train_batch_size=32,   # bigger batch = more positives per batch for contrastive loss
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=20,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    dataloader_drop_last=True   # avoid tiny last batch with too few positives for contrastive loss
)

trainer = ContrastiveTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=default_data_collator)
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  953MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch,Training Loss,Validation Loss
1,4.601573,4.549262
2,4.476632,4.433892
3,4.362216,4.391857
4,4.305640,4.372235
5,4.191074,4.428731
6,4.185942,4.421517


TrainOutput(global_step=552, training_loss=4.37170914636142, metrics={'train_runtime': 430.2195, 'train_samples_per_second': 41.407, 'train_steps_per_second': 1.283, 'total_flos': 0.0, 'train_loss': 4.37170914636142, 'epoch': 6.0})

### Quick sanity check: implicit-hate recall at default threshold (full tuning is Day 2)

In [11]:
model.eval()
device = next(model.parameters()).device
loader = torch.utils.data.DataLoader(test_ds, batch_size=32, collate_fn=default_data_collator)

probs1, true1, true3 = [], [], []
with torch.no_grad():
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        p = torch.softmax(out['logits1'], dim=-1)[:, HATE_IDX]
        probs1.extend(p.cpu().numpy()); true1.extend(batch['label1'].cpu().numpy()); true3.extend(batch['label3'].cpu().numpy())
probs1, true1, true3 = np.array(probs1), np.array(true1), np.array(true3)
pred1 = (probs1 >= 0.5).astype(int)

print(classification_report((true1==HATE_IDX).astype(int), pred1, target_names=['NON-HATE','HATE']))
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (true1==HATE_IDX) & (true3==idx)
    print(cat, 'recall @0.5:', pred1[m].mean(), 'n=', m.sum())

              precision    recall  f1-score   support

    NON-HATE       0.76      0.79      0.78       302
        HATE       0.80      0.78      0.79       335

    accuracy                           0.78       637
   macro avg       0.78      0.78      0.78       637
weighted avg       0.78      0.78      0.78       637

EXPLICIT recall @0.5: 0.8819444444444444 n= 144
IMPLICIT recall @0.5: 0.7058823529411765 n= 187


In [12]:
torch.save(model.state_dict(), 'contrastive_muril_day1.pt')
tokenizer.save_pretrained('contrastive_muril_tok')
import pickle
with open('encoders_day1.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('saved — ready for Day 2 (TF-IDF ensemble + stacking + threshold tuning + triage)')

saved — ready for Day 2 (TF-IDF ensemble + stacking + threshold tuning + triage)


In [13]:
!zip -r tok.zip contrastive_muril_tok
from google.colab import files
for f in ['contrastive_muril_day1.pt', 'encoders_day1.pkl', 'train_clean.csv', 'val_clean.csv', 'test_clean.csv', 'tok.zip']:
    files.download(f)

  adding: contrastive_muril_tok/ (stored 0%)
  adding: contrastive_muril_tok/tokenizer_config.json (deflated 47%)
  adding: contrastive_muril_tok/tokenizer.json (deflated 70%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import os
if not os.path.exists('contrastive_muril_day1.pt'):
    from google.colab import files
    print('Upload: contrastive_muril_day1.pt, encoders_day1.pkl, train_clean.csv, val_clean.csv, test_clean.csv, tok.zip')
    files.upload()
    if os.path.exists('tok.zip') and not os.path.exists('contrastive_muril_tok'):
        import zipfile
        with zipfile.ZipFile('tok.zip') as z: z.extractall('.')
else:
    print('Found existing Day 1 files in this session — no upload needed.')

Found existing Day 1 files in this session — no upload needed.


In [15]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, pickle
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, fbeta_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from transformers import AutoTokenizer, AutoModel, default_data_collator
from datasets import Dataset

MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128

with open('encoders_day1.pkl', 'rb') as f:
    encoders = pickle.load(f)
num_classes = {
    'Level1_Class': len(encoders['Level1_Class'].classes_),
    'Level2_Target': len(encoders['Level2_Target'].classes_),
    'Level3_Category': len(encoders['Level3_Category'].classes_),
}
HATE_IDX = list(encoders['Level1_Class'].classes_).index('HATE')
IMPLICIT_IDX = list(encoders['Level3_Category'].classes_).index('IMPLICIT')

train_df = pd.read_csv('train_clean.csv')
val_df = pd.read_csv('val_clean.csv')
test_df = pd.read_csv('test_clean.csv')
print(len(train_df), len(val_df), len(test_df))

2969 636 637


In [16]:
class ContrastiveMuril(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({
            'label1': nn.Linear(hidden, num_classes['Level1_Class']),
            'label2': nn.Linear(hidden, num_classes['Level2_Target']),
            'label3': nn.Linear(hidden, num_classes['Level3_Category']),
        })
        self.proj_head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Linear(256, 128))

    def forward(self, input_ids, attention_mask, **kw):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        logits = {k: h(cls) for k, h in self.heads.items()}
        return {'logits1': logits['label1'], 'logits2': logits['label2'], 'logits3': logits['label3']}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ContrastiveMuril(MODEL_NAME, num_classes)
model.load_state_dict(torch.load('contrastive_muril_day1.pt', map_location=device))
model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained('contrastive_muril_tok')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
def build_ds(d):
    ds = Dataset.from_pandas(d.reset_index(drop=True))
    def enc(batch):
        tok = tokenizer(batch['Input_Text'], padding='max_length', truncation=True, max_length=MAX_LEN)
        tok['label1'] = encoders['Level1_Class'].transform(batch['Level1_Class'])
        tok['label3'] = encoders['Level3_Category'].transform(batch['Level3_Category'])
        return tok
    return ds.map(enc, batched=True, remove_columns=ds.column_names)

val_ds, test_ds = build_ds(val_df), build_ds(test_df)

def get_muril_probs(ds):
    loader = torch.utils.data.DataLoader(ds, batch_size=32, collate_fn=default_data_collator)
    probs, true1, true3 = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            p = torch.softmax(out['logits1'], dim=-1)[:, HATE_IDX]
            probs.extend(p.cpu().numpy()); true1.extend(batch['label1'].cpu().numpy()); true3.extend(batch['label3'].cpu().numpy())
    return np.array(probs), np.array(true1), np.array(true3)

val_muril_probs, val_true1, val_true3 = get_muril_probs(val_ds)
test_muril_probs, test_true1, test_true3 = get_muril_probs(test_ds)

Map:   0%|          | 0/636 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [18]:
y_train = (train_df['Level1_Class'] == 'HATE').astype(int)
y_val = (val_df['Level1_Class'] == 'HATE').astype(int)
y_test = (test_df['Level1_Class'] == 'HATE').astype(int)

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), max_features=20000)
X_train = tfidf.fit_transform(train_df['Input_Text'])
X_val = tfidf.transform(val_df['Input_Text'])
X_test = tfidf.transform(test_df['Input_Text'])

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
lgbm.fit(X_train, y_train)

val_tfidf_probs = lgbm.predict_proba(X_val)[:, 1]
test_tfidf_probs = lgbm.predict_proba(X_test)[:, 1]
print('TF-IDF+LGBM val AUC:', roc_auc_score(y_val, val_tfidf_probs))

[LightGBM] [Info] Number of positive: 1561, number of negative: 1408
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.246684 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 181233
[LightGBM] [Info] Number of data points in the train set: 2969, number of used features: 5847
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.525766 -> initscore=0.103156
[LightGBM] [Info] Start training from score 0.103156


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


TF-IDF+LGBM val AUC: 0.8730221675853591


In [19]:
stack_X_val = np.column_stack([val_muril_probs, val_tfidf_probs])
stack_X_test = np.column_stack([test_muril_probs, test_tfidf_probs])

stacker = LogisticRegression()
stacker.fit(stack_X_val, y_val)   # trained on val (held out from MuRIL training), evaluated on test

val_stack_probs = stacker.predict_proba(stack_X_val)[:, 1]
test_stack_probs = stacker.predict_proba(stack_X_test)[:, 1]
print('stacker coefs (muril, tfidf):', stacker.coef_)

stacker coefs (muril, tfidf): [[4.45226946 3.52180571]]


In [20]:
prec, rec, thresh = precision_recall_curve(y_val, val_stack_probs)
f2_scores = [fbeta_score(y_val, val_stack_probs >= t, beta=2) for t in thresh]
best_idx = int(np.argmax(f2_scores))
BEST_THRESHOLD = thresh[best_idx]
print('best threshold (F2-optimal):', BEST_THRESHOLD, '| F2:', f2_scores[best_idx])

for t, label in [(0.5, 'default 0.5'), (BEST_THRESHOLD, 'tuned')]:
    implicit_mask = (val_true1 == HATE_IDX) & (val_true3 == IMPLICIT_IDX)
    recall_implicit = (val_stack_probs[implicit_mask] >= t).mean()
    print(f'{label}: implicit-hate recall (stacked) = {recall_implicit:.3f}')

best threshold (F2-optimal): 0.20441650434479783 | F2: 0.8986562150055991
default 0.5: implicit-hate recall (stacked) = 0.758
tuned: implicit-hate recall (stacked) = 0.941


In [21]:
test_pred_stack = (test_stack_probs >= BEST_THRESHOLD).astype(int)

print('=== STACKED (MuRIL + TF-IDF) ===')
print(classification_report(y_test, test_pred_stack, target_names=['NON-HATE','HATE']))
print(confusion_matrix(y_test, test_pred_stack))
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (test_true1==HATE_IDX) & (test_true3==idx)
    print(cat, 'recall:', test_pred_stack[m].mean(), 'n=', m.sum())

print('\n=== MuRIL-only @ same threshold, for comparison ===')
test_pred_muril_only = (test_muril_probs >= BEST_THRESHOLD).astype(int)
print(classification_report(y_test, test_pred_muril_only, target_names=['NON-HATE','HATE']))

=== STACKED (MuRIL + TF-IDF) ===
              precision    recall  f1-score   support

    NON-HATE       0.87      0.49      0.63       302
        HATE       0.67      0.93      0.78       335

    accuracy                           0.72       637
   macro avg       0.77      0.71      0.70       637
weighted avg       0.77      0.72      0.71       637

[[148 154]
 [ 22 313]]
EXPLICIT recall: 0.9722222222222222 n= 144
IMPLICIT recall: 0.9144385026737968 n= 187

=== MuRIL-only @ same threshold, for comparison ===
              precision    recall  f1-score   support

    NON-HATE       0.00      0.00      0.00       302
        HATE       0.53      1.00      0.69       335

    accuracy                           0.53       637
   macro avg       0.26      0.50      0.34       637
weighted avg       0.28      0.53      0.36       637



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
REVIEW_BAND = 0.15

def triage(prob):
    if prob >= BEST_THRESHOLD + REVIEW_BAND: return 'HIGH_CONF_HATE'
    if prob <= BEST_THRESHOLD - REVIEW_BAND: return 'HIGH_CONF_NONHATE'
    return 'REVIEW'

nil_idx = list(encoders['Level3_Category'].classes_).index('Nil'.upper()) if 'NIL' in [c.upper() for c in encoders['Level3_Category'].classes_] else None

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
    muril_prob = torch.softmax(out['logits1'], dim=-1)[0, HATE_IDX].item()
    tfidf_prob = lgbm.predict_proba(tfidf.transform([text]))[0, 1]
    stacked_prob = stacker.predict_proba([[muril_prob, tfidf_prob]])[0, 1]

    label = 'HATE' if stacked_prob >= BEST_THRESHOLD else 'NON-HATE'
    result = {'Level1_Class': label, 'hate_prob': round(float(stacked_prob), 3), 'triage': triage(stacked_prob)}
    if label == 'HATE':
        l3_logits = out['logits3'].clone()
        if nil_idx is not None:
            l3_logits[0, nil_idx] = -1e9   # exclude NIL from consideration when we already know it's HATE
        l2 = out['logits2'].argmax(-1).item(); l3 = l3_logits.argmax(-1).item()
        result['Level2_Target'] = encoders['Level2_Target'].inverse_transform([l2])[0]
        result['Level3_Category'] = encoders['Level3_Category'].inverse_transform([l3])[0]
    else:
        result['Level2_Target'] = 'Nil'; result['Level3_Category'] = 'Nil'
    return result

samples = [
    "இன்று வானிலை மிகவும் அருமையாக இருக்கிறது",
    "நீ ஒரு முட்டாள் பயலு, உன்னால ஒன்னுமே முடியாது",
    "இந்த மதத்தவங்க எல்லாம் நாட்டை அழிக்குறாங்க",
    "பொண்ணுங்க வீட்ல தான் இருக்கணும், வெளில வர கூடாது",
]
for s in samples:
    print(predict(s), '|', s)

{'Level1_Class': 'NON-HATE', 'hate_prob': 0.107, 'triage': 'REVIEW', 'Level2_Target': 'Nil', 'Level3_Category': 'Nil'} | இன்று வானிலை மிகவும் அருமையாக இருக்கிறது
{'Level1_Class': 'HATE', 'hate_prob': 0.896, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against Women', 'Level3_Category': 'EXPLICIT'} | நீ ஒரு முட்டாள் பயலு, உன்னால ஒன்னுமே முடியாது


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'Level1_Class': 'HATE', 'hate_prob': 0.729, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against Women', 'Level3_Category': 'EXPLICIT'} | இந்த மதத்தவங்க எல்லாம் நாட்டை அழிக்குறாங்க
{'Level1_Class': 'HATE', 'hate_prob': 0.25, 'triage': 'REVIEW', 'Level2_Target': 'Hate against Men', 'Level3_Category': 'EXPLICIT'} | பொண்ணுங்க வீட்ல தான் இருக்கணும், வெளில வர கூடாது


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [23]:
import pickle
with open('tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('lgbm_model.pkl', 'wb') as f: pickle.dump(lgbm, f)
with open('stacker.pkl', 'wb') as f: pickle.dump({'model': stacker, 'threshold': BEST_THRESHOLD}, f)
print('saved — ready for Day 3 (error analysis, SHAP/attention viz, writeup)')

saved — ready for Day 3 (error analysis, SHAP/attention viz, writeup)


DAY 2 Trial 2

In [25]:
import os
if not os.path.exists('contrastive_muril_day1.pt'):
    from google.colab import files
    print('Upload: contrastive_muril_day1.pt, encoders_day1.pkl, train_clean.csv, val_clean.csv, test_clean.csv, tok.zip')
    files.upload()
    if os.path.exists('tok.zip') and not os.path.exists('contrastive_muril_tok'):
        import zipfile
        with zipfile.ZipFile('tok.zip') as z: z.extractall('.')
else:
    print('Found existing Day 1 files in this session — no upload needed.')

Found existing Day 1 files in this session — no upload needed.


In [26]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.nn.functional as F, pickle
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, fbeta_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
from transformers import AutoTokenizer, AutoModel, default_data_collator
from datasets import Dataset

MODEL_NAME = "google/muril-base-cased"
MAX_LEN = 128

with open('encoders_day1.pkl', 'rb') as f:
    encoders = pickle.load(f)
num_classes = {
    'Level1_Class': len(encoders['Level1_Class'].classes_),
    'Level2_Target': len(encoders['Level2_Target'].classes_),
    'Level3_Category': len(encoders['Level3_Category'].classes_),
}
HATE_IDX = list(encoders['Level1_Class'].classes_).index('HATE')
IMPLICIT_IDX = list(encoders['Level3_Category'].classes_).index('IMPLICIT')

train_df = pd.read_csv('train_clean.csv')
val_df = pd.read_csv('val_clean.csv')
test_df = pd.read_csv('test_clean.csv')
print(len(train_df), len(val_df), len(test_df))

2969 636 637


In [27]:
class ContrastiveMuril(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({
            'label1': nn.Linear(hidden, num_classes['Level1_Class']),
            'label2': nn.Linear(hidden, num_classes['Level2_Target']),
            'label3': nn.Linear(hidden, num_classes['Level3_Category']),
        })
        self.proj_head = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Linear(256, 128))

    def forward(self, input_ids, attention_mask, **kw):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0])
        logits = {k: h(cls) for k, h in self.heads.items()}
        return {'logits1': logits['label1'], 'logits2': logits['label2'], 'logits3': logits['label3']}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ContrastiveMuril(MODEL_NAME, num_classes)
model.load_state_dict(torch.load('contrastive_muril_day1.pt', map_location=device))
model.to(device).eval()

tokenizer = AutoTokenizer.from_pretrained('contrastive_muril_tok')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
def build_ds(d):
    ds = Dataset.from_pandas(d.reset_index(drop=True))
    def enc(batch):
        tok = tokenizer(batch['Input_Text'], padding='max_length', truncation=True, max_length=MAX_LEN)
        tok['label1'] = encoders['Level1_Class'].transform(batch['Level1_Class'])
        tok['label3'] = encoders['Level3_Category'].transform(batch['Level3_Category'])
        return tok
    return ds.map(enc, batched=True, remove_columns=ds.column_names)

val_ds, test_ds = build_ds(val_df), build_ds(test_df)

def get_muril_probs(ds):
    loader = torch.utils.data.DataLoader(ds, batch_size=32, collate_fn=default_data_collator)
    probs, true1, true3 = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            p = torch.softmax(out['logits1'], dim=-1)[:, HATE_IDX]
            probs.extend(p.cpu().numpy()); true1.extend(batch['label1'].cpu().numpy()); true3.extend(batch['label3'].cpu().numpy())
    return np.array(probs), np.array(true1), np.array(true3)

val_muril_probs, val_true1, val_true3 = get_muril_probs(val_ds)
test_muril_probs, test_true1, test_true3 = get_muril_probs(test_ds)

Map:   0%|          | 0/636 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

In [29]:
y_train = (train_df['Level1_Class'] == 'HATE').astype(int)
y_val = (val_df['Level1_Class'] == 'HATE').astype(int)
y_test = (test_df['Level1_Class'] == 'HATE').astype(int)

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), max_features=20000)
X_train = tfidf.fit_transform(train_df['Input_Text'])
X_val = tfidf.transform(val_df['Input_Text'])
X_test = tfidf.transform(test_df['Input_Text'])

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
lgbm.fit(X_train, y_train)

val_tfidf_probs = lgbm.predict_proba(X_val)[:, 1]
test_tfidf_probs = lgbm.predict_proba(X_test)[:, 1]
print('TF-IDF+LGBM val AUC:', roc_auc_score(y_val, val_tfidf_probs))

[LightGBM] [Info] Number of positive: 1561, number of negative: 1408
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.280054 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 181233
[LightGBM] [Info] Number of data points in the train set: 2969, number of used features: 5847
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.525766 -> initscore=0.103156
[LightGBM] [Info] Start training from score 0.103156
TF-IDF+LGBM val AUC: 0.8730221675853591


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [30]:
stack_X_val = np.column_stack([val_muril_probs, val_tfidf_probs])
stack_X_test = np.column_stack([test_muril_probs, test_tfidf_probs])

stacker = LogisticRegression()
stacker.fit(stack_X_val, y_val)   # trained on val (held out from MuRIL training), evaluated on test

val_stack_probs = stacker.predict_proba(stack_X_val)[:, 1]
test_stack_probs = stacker.predict_proba(stack_X_test)[:, 1]
print('stacker coefs (muril, tfidf):', stacker.coef_)

stacker coefs (muril, tfidf): [[4.45226946 3.52180571]]


In [31]:
prec, rec, thresh = precision_recall_curve(y_val, val_stack_probs)
f2_scores = [fbeta_score(y_val, val_stack_probs >= t, beta=2) for t in thresh]
best_idx = int(np.argmax(f2_scores))
BEST_THRESHOLD = thresh[best_idx]
print('best threshold (F2-optimal):', BEST_THRESHOLD, '| F2:', f2_scores[best_idx])

for t, label in [(0.5, 'default 0.5'), (BEST_THRESHOLD, 'tuned')]:
    implicit_mask = (val_true1 == HATE_IDX) & (val_true3 == IMPLICIT_IDX)
    recall_implicit = (val_stack_probs[implicit_mask] >= t).mean()
    print(f'{label}: implicit-hate recall (stacked) = {recall_implicit:.3f}')

best threshold (F2-optimal): 0.20441650434479783 | F2: 0.8986562150055991
default 0.5: implicit-hate recall (stacked) = 0.758
tuned: implicit-hate recall (stacked) = 0.941


In [32]:
test_pred_stack = (test_stack_probs >= BEST_THRESHOLD).astype(int)

print('=== STACKED (MuRIL + TF-IDF) ===')
print(classification_report(y_test, test_pred_stack, target_names=['NON-HATE','HATE']))
print(confusion_matrix(y_test, test_pred_stack))
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (test_true1==HATE_IDX) & (test_true3==idx)
    print(cat, 'recall:', test_pred_stack[m].mean(), 'n=', m.sum())

print('\n=== MuRIL-only @ same threshold, for comparison ===')
test_pred_muril_only = (test_muril_probs >= BEST_THRESHOLD).astype(int)
print(classification_report(y_test, test_pred_muril_only, target_names=['NON-HATE','HATE']))

=== STACKED (MuRIL + TF-IDF) ===
              precision    recall  f1-score   support

    NON-HATE       0.87      0.49      0.63       302
        HATE       0.67      0.93      0.78       335

    accuracy                           0.72       637
   macro avg       0.77      0.71      0.70       637
weighted avg       0.77      0.72      0.71       637

[[148 154]
 [ 22 313]]
EXPLICIT recall: 0.9722222222222222 n= 144
IMPLICIT recall: 0.9144385026737968 n= 187

=== MuRIL-only @ same threshold, for comparison ===
              precision    recall  f1-score   support

    NON-HATE       0.00      0.00      0.00       302
        HATE       0.53      1.00      0.69       335

    accuracy                           0.53       637
   macro avg       0.26      0.50      0.34       637
weighted avg       0.28      0.53      0.36       637



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
REVIEW_BAND = 0.15

def triage(prob):
    if prob >= BEST_THRESHOLD + REVIEW_BAND: return 'HIGH_CONF_HATE'
    if prob <= BEST_THRESHOLD - REVIEW_BAND: return 'HIGH_CONF_NONHATE'
    return 'REVIEW'

def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
    muril_prob = torch.softmax(out['logits1'], dim=-1)[0, HATE_IDX].item()
    tfidf_prob = lgbm.predict_proba(tfidf.transform([text]))[0, 1]
    stacked_prob = stacker.predict_proba([[muril_prob, tfidf_prob]])[0, 1]

    label = 'HATE' if stacked_prob >= BEST_THRESHOLD else 'NON-HATE'
    result = {'Level1_Class': label, 'hate_prob': round(float(stacked_prob), 3), 'triage': triage(stacked_prob)}
    if label == 'HATE':
        l2 = out['logits2'].argmax(-1).item(); l3 = out['logits3'].argmax(-1).item()
        result['Level2_Target'] = encoders['Level2_Target'].inverse_transform([l2])[0]
        result['Level3_Category'] = encoders['Level3_Category'].inverse_transform([l3])[0]
    else:
        result['Level2_Target'] = 'Nil'; result['Level3_Category'] = 'Nil'
    return result

samples = [
    "இன்று வானிலை மிகவும் அருமையாக இருக்கிறது",
    "நீ ஒரு முட்டாள் பயலு, உன்னால ஒன்னுமே முடியாது",
    "இந்த மதத்தவங்க எல்லாம் நாட்டை அழிக்குறாங்க",
    "பொண்ணுங்க வீட்ல தான் இருக்கணும், வெளில வர கூடாது",
]
for s in samples:
    print(predict(s), '|', s)

{'Level1_Class': 'NON-HATE', 'hate_prob': 0.107, 'triage': 'REVIEW', 'Level2_Target': 'Nil', 'Level3_Category': 'Nil'} | இன்று வானிலை மிகவும் அருமையாக இருக்கிறது
{'Level1_Class': 'HATE', 'hate_prob': 0.896, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against Women', 'Level3_Category': 'EXPLICIT'} | நீ ஒரு முட்டாள் பயலு, உன்னால ஒன்னுமே முடியாது


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'Level1_Class': 'HATE', 'hate_prob': 0.729, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against Women', 'Level3_Category': 'NIL'} | இந்த மதத்தவங்க எல்லாம் நாட்டை அழிக்குறாங்க
{'Level1_Class': 'HATE', 'hate_prob': 0.25, 'triage': 'REVIEW', 'Level2_Target': 'Hate against Men', 'Level3_Category': 'NIL'} | பொண்ணுங்க வீட்ல தான் இருக்கணும், வெளில வர கூடாது


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [34]:
from sklearn.linear_model import LogisticRegression as LR

@torch.no_grad()
def get_embeddings(texts, batch_size=32):
    embs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = list(texts[i:i+batch_size])
        inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        out = model.backbone(**inputs)
        cls = out.last_hidden_state[:, 0]
        embs.append(cls.cpu().numpy())
    return np.concatenate(embs, axis=0)

train_emb = get_embeddings(train_df['Input_Text'].tolist())
test_emb = get_embeddings(test_df['Input_Text'].tolist())
print('embeddings shape:', train_emb.shape)

embeddings shape: (2969, 768)


In [35]:
train_hate_mask = (train_df['Level1_Class'] == 'HATE') & \
                   (train_df['Level2_Target'] != 'Nil') & \
                   (train_df['Level3_Category'].str.upper() != 'NIL')
test_hate_mask = (test_df['Level1_Class'] == 'HATE') & \
                  (test_df['Level2_Target'] != 'Nil') & \
                  (test_df['Level3_Category'].str.upper() != 'NIL')

print('dropped', (train_df['Level1_Class']=='HATE').sum() - train_hate_mask.sum(), 'inconsistently-labeled HATE rows from training')

y2_train = encoders['Level2_Target'].transform(train_df.loc[train_hate_mask, 'Level2_Target'])
y3_train = encoders['Level3_Category'].transform(train_df.loc[train_hate_mask, 'Level3_Category'])

head2 = LR(max_iter=1000, class_weight='balanced').fit(train_emb[train_hate_mask], y2_train)
head3 = LR(max_iter=1000, class_weight='balanced').fit(train_emb[train_hate_mask], y3_train)

y2_test = encoders['Level2_Target'].transform(test_df.loc[test_hate_mask, 'Level2_Target'])
y3_test = encoders['Level3_Category'].transform(test_df.loc[test_hate_mask, 'Level3_Category'])
pred2_test = head2.predict(test_emb[test_hate_mask])
pred3_test = head3.predict(test_emb[test_hate_mask])

print('--- Level3 (recalibrated head, clean HATE rows only) ---')
print(classification_report(y3_test, pred3_test, labels=np.unique(y3_test), target_names=encoders['Level3_Category'].classes_[np.unique(y3_test)], zero_division=0))
print('--- Level2 (recalibrated head, clean HATE rows only) ---')
print(classification_report(y2_test, pred2_test, labels=np.unique(y2_test), target_names=encoders['Level2_Target'].classes_[np.unique(y2_test)], zero_division=0))

dropped 39 inconsistently-labeled HATE rows from training
--- Level3 (recalibrated head, clean HATE rows only) ---
              precision    recall  f1-score   support

    EXPLICIT       0.65      0.69      0.67       141
    IMPLICIT       0.75      0.72      0.73       183

    accuracy                           0.70       324
   macro avg       0.70      0.70      0.70       324
weighted avg       0.71      0.70      0.70       324

--- Level2 (recalibrated head, clean HATE rows only) ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.13      0.60      0.21        25
   Hate against Men       0.35      0.50      0.41        86
 Hate against Women       0.64      0.22      0.33       166
 No Target- Profane       0.20      0.11      0.14        47

           accuracy                           0.31       324
          macro avg       0.33      0.36      0.27       324
       weighted avg       0.46      0.31      0.31       324



In [36]:
boot_thresholds = []
rng = np.random.default_rng(SEED)
for _ in range(200):
    idx = rng.integers(0, len(val_stack_probs), len(val_stack_probs))
    yb, pb = y_val.values[idx] if hasattr(y_val, 'values') else np.array(y_val)[idx], val_stack_probs[idx]
    p, r, t = precision_recall_curve(yb, pb)
    if len(t) == 0: continue
    f2 = [fbeta_score(yb, pb >= th, beta=2, zero_division=0) for th in t]
    boot_thresholds.append(t[int(np.argmax(f2))])

boot_thresholds = np.array(boot_thresholds)
print(f'threshold: mean={boot_thresholds.mean():.3f}  std={boot_thresholds.std():.3f}  [p5={np.percentile(boot_thresholds,5):.3f}, p95={np.percentile(boot_thresholds,95):.3f}]')
print(f'current BEST_THRESHOLD={BEST_THRESHOLD:.3f} — ', 'stable' if boot_thresholds.std() < 0.05 else 'UNSTABLE, consider using the bootstrap mean instead')

threshold: mean=0.211  std=0.036  [p5=0.143, p95=0.249]
current BEST_THRESHOLD=0.204 —  stable


In [37]:
def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
        cls_emb = model.backbone(**inputs).last_hidden_state[:, 0].cpu().numpy()
    muril_prob = torch.softmax(out['logits1'], dim=-1)[0, HATE_IDX].item()
    tfidf_prob = lgbm.predict_proba(tfidf.transform([text]))[0, 1]
    stacked_prob = stacker.predict_proba([[muril_prob, tfidf_prob]])[0, 1]

    label = 'HATE' if stacked_prob >= BEST_THRESHOLD else 'NON-HATE'
    result = {'Level1_Class': label, 'hate_prob': round(float(stacked_prob), 3), 'triage': triage(stacked_prob)}
    if label == 'HATE':
        l2 = head2.predict(cls_emb)[0]
        l3 = head3.predict(cls_emb)[0]
        result['Level2_Target'] = encoders['Level2_Target'].inverse_transform([l2])[0]
        result['Level3_Category'] = encoders['Level3_Category'].inverse_transform([l3])[0]
    else:
        result['Level2_Target'] = 'Nil'; result['Level3_Category'] = 'Nil'
    return result

for s in samples:
    print(predict(s), '|', s)

{'Level1_Class': 'NON-HATE', 'hate_prob': 0.107, 'triage': 'REVIEW', 'Level2_Target': 'Nil', 'Level3_Category': 'Nil'} | இன்று வானிலை மிகவும் அருமையாக இருக்கிறது
{'Level1_Class': 'HATE', 'hate_prob': 0.896, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against Men', 'Level3_Category': 'EXPLICIT'} | நீ ஒரு முட்டாள் பயலு, உன்னால ஒன்னுமே முடியாது
{'Level1_Class': 'HATE', 'hate_prob': 0.729, 'triage': 'HIGH_CONF_HATE', 'Level2_Target': 'Hate against LGBTQ+', 'Level3_Category': 'IMPLICIT'} | இந்த மதத்தவங்க எல்லாம் நாட்டை அழிக்குறாங்க
{'Level1_Class': 'HATE', 'hate_prob': 0.25, 'triage': 'REVIEW', 'Level2_Target': 'Hate against LGBTQ+', 'Level3_Category': 'IMPLICIT'} | பொண்ணுங்க வீட்ல தான் இருக்கணும், வெளில வர கூடாது


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [38]:
import pickle
with open('tfidf_vectorizer.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('lgbm_model.pkl', 'wb') as f: pickle.dump(lgbm, f)
with open('stacker.pkl', 'wb') as f: pickle.dump({'model': stacker, 'threshold': BEST_THRESHOLD}, f)
with open('recalibrated_heads.pkl', 'wb') as f: pickle.dump({'head2': head2, 'head3': head3}, f)
print('saved — ready for Day 3 (error analysis, SHAP/attention viz, writeup)')

saved — ready for Day 3 (error analysis, SHAP/attention viz, writeup)


In [39]:
from google.colab import files

save_list = [
    'contrastive_muril_day1.pt', 'encoders_day1.pkl',
    'train_clean.csv', 'val_clean.csv', 'test_clean.csv',
    'tfidf_vectorizer.pkl', 'lgbm_model.pkl', 'stacker.pkl', 'recalibrated_heads.pkl'
]

!zip -r tok.zip contrastive_muril_tok
save_list.append('tok.zip')

import os
for f in save_list:
    if os.path.exists(f):
        files.download(f)
    else:
        print('MISSING (skip):', f)

updating: contrastive_muril_tok/ (stored 0%)
updating: contrastive_muril_tok/tokenizer_config.json (deflated 47%)
updating: contrastive_muril_tok/tokenizer.json (deflated 70%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (11 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:

stack_X_test = np.column_stack([test_muril_probs, test_tfidf_probs])
test_stack_probs = stacker.predict_proba(stack_X_test)[:, 1]
test_pred_stack = (test_stack_probs >= BEST_THRESHOLD).astype(int)
y_test = (test_df['Level1_Class'] == 'HATE').astype(int).values

In [41]:
print("="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
print(classification_report(y_test, test_pred_stack, target_names=['NON-HATE','HATE']))
print(confusion_matrix(y_test, test_pred_stack))

print("\n--- Level1 recall by Level3 category ---")
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (test_true1==HATE_IDX) & (test_true3==idx)
    print(cat, 'recall:', round(test_pred_stack[m].mean(), 3), 'n=', m.sum())

print("\n--- Level3 classification (recalibrated head, clean HATE rows) ---")
print(classification_report(y3_test, pred3_test, labels=np.unique(y3_test), target_names=encoders['Level3_Category'].classes_[np.unique(y3_test)], zero_division=0))

print("\n--- Level2 classification (recalibrated head, clean HATE rows) ---")
print(classification_report(y2_test, pred2_test, labels=np.unique(y2_test), target_names=encoders['Level2_Target'].classes_[np.unique(y2_test)], zero_division=0))

FINAL RESULTS SUMMARY
              precision    recall  f1-score   support

    NON-HATE       0.87      0.49      0.63       302
        HATE       0.67      0.93      0.78       335

    accuracy                           0.72       637
   macro avg       0.77      0.71      0.70       637
weighted avg       0.77      0.72      0.71       637

[[148 154]
 [ 22 313]]

--- Level1 recall by Level3 category ---
EXPLICIT recall: 0.972 n= 144
IMPLICIT recall: 0.914 n= 187

--- Level3 classification (recalibrated head, clean HATE rows) ---
              precision    recall  f1-score   support

    EXPLICIT       0.65      0.69      0.67       141
    IMPLICIT       0.75      0.72      0.73       183

    accuracy                           0.70       324
   macro avg       0.70      0.70      0.70       324
weighted avg       0.71      0.70      0.70       324


--- Level2 classification (recalibrated head, clean HATE rows) ---
                     precision    recall  f1-score   support

H

In [42]:
test_texts = test_df['Input_Text'].values
implicit_mask = (test_true1==HATE_IDX) & (test_true3==IMPLICIT_IDX)

caught = np.where(implicit_mask & (test_pred_stack==1))[0]
missed = np.where(implicit_mask & (test_pred_stack==0))[0]

print(f"Implicit hate correctly caught: {len(caught)}/{implicit_mask.sum()}")
print("\n--- 5 examples CORRECTLY caught (no lexical trigger, model used context) ---")
for i in caught[:5]:
    print(f"  [{test_stack_probs[i]:.3f}] {test_texts[i]}")

print(f"\n--- 5 examples STILL MISSED (limitation to disclose) ---")
for i in missed[:5]:
    print(f"  [{test_stack_probs[i]:.3f}] {test_texts[i]}")

Implicit hate correctly caught: 171/187

--- 5 examples CORRECTLY caught (no lexical trigger, model used context) ---
  [0.320] Dei dei Avan oru athu Ne oru ithu  ponga da deiii
  [0.781] இந்த அம்மா ஷகிலா பெரிய பத்தினி பணத்துகாக சினிமாவுக்கு வரல சமுக சேவை செய்ய வந்துச்சு பொதுவெளியில் சகமனிதரை எப்படி ஓய்வு வேண்டும்னு தெரியல துது
  [0.882] சகிலா வந்தனா காரி மூஞ்சிலேயே துப்பி போடுவேன்😮
  [0.898] நல்லா ஓங்கி மூஞ்சிலே குத்தனும் போல இருக்கு அப்படி கோபம் வருது இத பாத்தா இதெல்லாம் ஒரு பொண்ணு சிரிப்பு ஒன்னு தான் கேடு இதுக்கு கன்றாவி
  [0.236] total tamilnadu peoples be like::;கண்டா வரச் சொல்லுங்க கண்டாரஓலி கார்த்திய கையோடு கூட்டி வாருங்க

--- 5 examples STILL MISSED (limitation to disclose) ---
  [0.147] இன்னேரம் அந்த கார்த்தி Gazette ல பேர மாத்திட்டு கேரளாவுக்குப் போய் தலைமறைவாயிருப்பான்
  [0.142] திருவிழா கூட்டத்தில் காணாமல் போன கார்த்தி எங்கிருந்தாலும் லட்சுமி ராமகிருஷ்ணன் இடம் வரவும்.
  [0.144] அந்த கோஜா  போய் தொலைட்டும்
  [0.179] இது போன்ற ஆட்களை மக்கள் புறக்கணிக்க வேண்டும்.உண்மையான talent

In [43]:
TARGET_KEYWORDS = ['பொண்', 'பெண்', 'ஆண்', 'மனைவி', 'மத', 'இந்து', 'முஸ்லிம்', 'கிறிஸ்', 'திருநங்கை', 'ஓரின']
nonhate_mask = (test_true1 != HATE_IDX)
has_kw = test_df['Input_Text'].apply(lambda t: any(k in t for k in TARGET_KEYWORDS)).values

fp_with_kw = test_pred_stack[nonhate_mask & has_kw].mean() if (nonhate_mask & has_kw).sum() else float('nan')
fp_without_kw = test_pred_stack[nonhate_mask & ~has_kw].mean() if (nonhate_mask & ~has_kw).sum() else float('nan')
print(f"False-positive rate, NON-HATE containing target-group words: {fp_with_kw:.3f} (n={(nonhate_mask & has_kw).sum()})")
print(f"False-positive rate, NON-HATE without target-group words:    {fp_without_kw:.3f} (n={(nonhate_mask & ~has_kw).sum()})")
print("Large gap here = model over-flags counter-speech/discussion about these groups as hate — worth naming as a limitation if found.")

False-positive rate, NON-HATE containing target-group words: 0.620 (n=50)
False-positive rate, NON-HATE without target-group words:    0.488 (n=252)
Large gap here = model over-flags counter-speech/discussion about these groups as hate — worth naming as a limitation if found.


In [44]:
import matplotlib.pyplot as plt

def visualize_attention(text, title=""):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.backbone(**inputs, output_attentions=True, attn_implementation="eager")
    last_layer_attn = out.attentions[-1][0]              # (heads, seq, seq)
    cls_attn = last_layer_attn[:, 0, :].mean(dim=0)       # avg heads, CLS row -> all tokens
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    scores = cls_attn.cpu().numpy()
    keep = [i for i, t in enumerate(tokens) if t not in ('[CLS]', '[SEP]', '[PAD]')]
    tokens = [tokens[i] for i in keep]
    scores = scores[keep]

    plt.figure(figsize=(max(6, len(tokens)*0.6), 2.5))
    plt.bar(range(len(tokens)), scores, color='crimson')
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right')
    plt.title(title or text)
    plt.ylabel('attention weight')
    plt.tight_layout()
    plt.show()

explicit_correct_idx = np.where((test_true3==1-IMPLICIT_IDX) & (test_pred_stack==1) & (test_true1==HATE_IDX))[0]
if len(explicit_correct_idx): visualize_attention(test_texts[explicit_correct_idx[0]], "Explicit hate — correctly caught")
if len(caught): visualize_attention(test_texts[caught[0]], "Implicit hate — correctly caught")
if len(missed): visualize_attention(test_texts[missed[0]], "Implicit hate — missed (limitation)")

[transformers] `sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


IndexError: tuple index out of range

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os, pickle
save_dir = '/content/drive/MyDrive/tamil_hate_project'
os.makedirs(save_dir, exist_ok=True)
with open(os.path.join(save_dir, 'final_artifacts.pkl'), 'wb') as f:
    pickle.dump({'tfidf': tfidf, 'lgbm': lgbm, 'stacker': stacker, 'threshold': BEST_THRESHOLD,
                 'head2': head2, 'head3': head3, 'encoders': encoders}, f)
shutil.copy('contrastive_muril_day1.pt', save_dir)
print('saved to Drive:', save_dir)

from google.colab import files
files.download('contrastive_muril_day1.pt')

In [45]:
# switch from F2 (recall-heavy) to F1 (balanced) threshold — cheap, instant, no retraining
prec, rec, thresh = precision_recall_curve(y_val, stacker.predict_proba(stack_X_val)[:,1])
f1_scores = [fbeta_score(y_val, stacker.predict_proba(stack_X_val)[:,1] >= t, beta=1) for t in thresh]
BEST_THRESHOLD_F1 = thresh[int(np.argmax(f1_scores))]
print('old (F2) threshold:', BEST_THRESHOLD, '-> new (F1) threshold:', BEST_THRESHOLD_F1)

BEST_THRESHOLD = BEST_THRESHOLD_F1   # overwrite — everything downstream (predict(), triage) uses this automatically

test_pred_stack = (test_stack_probs >= BEST_THRESHOLD).astype(int)
print(classification_report(y_test, test_pred_stack, target_names=['NON-HATE','HATE']))
print(confusion_matrix(y_test, test_pred_stack))
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (test_true1==HATE_IDX) & (test_true3==idx)
    print(cat, 'recall:', round(test_pred_stack[m].mean(),3), 'n=', m.sum())

old (F2) threshold: 0.20441650434479783 -> new (F1) threshold: 0.2932664988466742
              precision    recall  f1-score   support

    NON-HATE       0.83      0.59      0.69       302
        HATE       0.71      0.89      0.79       335

    accuracy                           0.75       637
   macro avg       0.77      0.74      0.74       637
weighted avg       0.77      0.75      0.74       637

[[178 124]
 [ 36 299]]
EXPLICIT recall: 0.944 n= 144
IMPLICIT recall: 0.861 n= 187


In [46]:
@torch.no_grad()
def get_embeddings_meanpool(texts, batch_size=32):
    embs = []
    for i in range(0, len(texts), batch_size):
        inputs = tokenizer(list(texts[i:i+batch_size]), return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        out = model.backbone(**inputs)
        mask = inputs['attention_mask'].unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        embs.append(pooled.cpu().numpy())
    return np.concatenate(embs, axis=0)

train_emb_mp = get_embeddings_meanpool(train_df['Input_Text'].tolist())
test_emb_mp = get_embeddings_meanpool(test_df['Input_Text'].tolist())

head2_mp = LR(max_iter=1000, class_weight='balanced').fit(train_emb_mp[train_hate_mask], y2_train)
head3_mp = LR(max_iter=1000, class_weight='balanced').fit(train_emb_mp[train_hate_mask], y3_train)

pred2_mp = head2_mp.predict(test_emb_mp[test_hate_mask])
pred3_mp = head3_mp.predict(test_emb_mp[test_hate_mask])
print('--- Level3, mean-pooled ---')
print(classification_report(y3_test, pred3_mp, labels=np.unique(y3_test), target_names=encoders['Level3_Category'].classes_[np.unique(y3_test)], zero_division=0))
print('--- Level2, mean-pooled ---')
print(classification_report(y2_test, pred2_mp, labels=np.unique(y2_test), target_names=encoders['Level2_Target'].classes_[np.unique(y2_test)], zero_division=0))

--- Level3, mean-pooled ---
              precision    recall  f1-score   support

    EXPLICIT       0.66      0.69      0.67       141
    IMPLICIT       0.75      0.72      0.74       183

    accuracy                           0.71       324
   macro avg       0.70      0.70      0.70       324
weighted avg       0.71      0.71      0.71       324

--- Level2, mean-pooled ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.14      0.64      0.23        25
   Hate against Men       0.35      0.50      0.41        86
 Hate against Women       0.65      0.22      0.33       166
 No Target- Profane       0.19      0.11      0.14        47

           accuracy                           0.31       324
          macro avg       0.33      0.37      0.28       324
       weighted avg       0.46      0.31      0.32       324



In [47]:
from sklearn.isotonic import IsotonicRegression

# diagnostic first: are muril and tfidf both biased high, or just one?
print('mean MuRIL prob | true HATE:', val_muril_probs[y_val==1].mean(), '| true NON-HATE:', val_muril_probs[y_val==0].mean())
print('mean TFIDF prob | true HATE:', val_tfidf_probs[y_val==1].mean(), '| true NON-HATE:', val_tfidf_probs[y_val==0].mean())

# calibrate each model's raw probabilities on val set before stacking
iso_muril = IsotonicRegression(out_of_bounds='clip').fit(val_muril_probs, y_val)
iso_tfidf = IsotonicRegression(out_of_bounds='clip').fit(val_tfidf_probs, y_val)

val_muril_cal = iso_muril.predict(val_muril_probs)
val_tfidf_cal = iso_tfidf.predict(val_tfidf_probs)
test_muril_cal = iso_muril.predict(test_muril_probs)
test_tfidf_cal = iso_tfidf.predict(test_tfidf_probs)

# refit stacker on calibrated inputs
stack_X_val_cal = np.column_stack([val_muril_cal, val_tfidf_cal])
stack_X_test_cal = np.column_stack([test_muril_cal, test_tfidf_cal])
stacker_cal = LR().fit(stack_X_val_cal, y_val)

# re-tune threshold (F1) on the calibrated stacker
prec, rec, thresh = precision_recall_curve(y_val, stacker_cal.predict_proba(stack_X_val_cal)[:,1])
f1_scores = [fbeta_score(y_val, stacker_cal.predict_proba(stack_X_val_cal)[:,1] >= t, beta=1) for t in thresh]
BEST_THRESHOLD_CAL = thresh[int(np.argmax(f1_scores))]

test_stack_probs_cal = stacker_cal.predict_proba(stack_X_test_cal)[:,1]
test_pred_cal = (test_stack_probs_cal >= BEST_THRESHOLD_CAL).astype(int)

print('\nthreshold:', BEST_THRESHOLD_CAL)
print(classification_report(y_test, test_pred_cal, target_names=['NON-HATE','HATE']))
print(confusion_matrix(y_test, test_pred_cal))
for cat, idx in [('EXPLICIT', 1-IMPLICIT_IDX), ('IMPLICIT', IMPLICIT_IDX)]:
    m = (test_true1==HATE_IDX) & (test_true3==idx)
    print(cat, 'recall:', round(test_pred_cal[m].mean(),3), 'n=', m.sum())

mean MuRIL prob | true HATE: 0.5889685 | true NON-HATE: 0.4729206
mean TFIDF prob | true HATE: 0.7567706928932147 | true NON-HATE: 0.2616291747340781

threshold: 0.41632148386329776
              precision    recall  f1-score   support

    NON-HATE       0.80      0.74      0.76       302
        HATE       0.78      0.83      0.80       335

    accuracy                           0.78       637
   macro avg       0.79      0.78      0.78       637
weighted avg       0.79      0.78      0.78       637

[[222  80]
 [ 57 278]]
EXPLICIT recall: 0.903 n= 144
IMPLICIT recall: 0.781 n= 187


In [48]:
stacker = stacker_cal
BEST_THRESHOLD = BEST_THRESHOLD_CAL

In [51]:
from google.colab import files
import pickle, os

# save everything into one bundle so you only download one file
with open('final_artifacts.pkl', 'wb') as f:
    pickle.dump({
        'tfidf': tfidf,
        'lgbm': lgbm,
        'stacker': stacker,          # already = stacker_cal after your reassignment
        'threshold': BEST_THRESHOLD, # already = BEST_THRESHOLD_CAL
        'head2': head2,
        'head3': head3,
        'encoders': encoders,
        'iso_muril': iso_muril,
        'iso_tfidf': iso_tfidf
    }, f)

save_list = [
    'contrastive_muril_day1.pt',
    'final_artifacts.pkl',
    'train_clean.csv', 'val_clean.csv', 'test_clean.csv'
]

!zip -r tok.zip contrastive_muril_tok
save_list.append('tok.zip')

for f in save_list:
    if os.path.exists(f):
        files.download(f)
    else:
        print('MISSING (skip):', f)

updating: contrastive_muril_tok/ (stored 0%)
updating: contrastive_muril_tok/tokenizer_config.json (deflated 47%)
updating: contrastive_muril_tok/tokenizer.json (deflated 70%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [52]:
print('stacker coefficients [muril, tfidf]:', stacker.coef_)
print('stacker intercept:', stacker.intercept_)

stacker coefficients [muril, tfidf]: [[3.36814594 2.91252747]]
stacker intercept: [-3.24099077]


In [53]:
from sklearn.pipeline import FeatureUnion

tfidf_word = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=10000, min_df=2)
X_train_word = tfidf_word.fit_transform(train_df['Input_Text'])
X_val_word = tfidf_word.transform(val_df['Input_Text'])
X_test_word = tfidf_word.transform(test_df['Input_Text'])

from scipy.sparse import hstack
X_train_combined = hstack([X_train, X_train_word])
X_val_combined = hstack([X_val, X_val_word])
X_test_combined = hstack([X_test, X_test_word])

lgbm_v2 = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42)
lgbm_v2.fit(X_train_combined, y_train)

val_tfidf_probs_v2 = lgbm_v2.predict_proba(X_val_combined)[:, 1]
test_tfidf_probs_v2 = lgbm_v2.predict_proba(X_test_combined)[:, 1]
print('word+char TF-IDF val AUC:', roc_auc_score(y_val, val_tfidf_probs_v2), '(compare to char-only:', roc_auc_score(y_val, val_tfidf_probs), ')')

[LightGBM] [Info] Number of positive: 1561, number of negative: 1408
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.275011 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 189219
[LightGBM] [Info] Number of data points in the train set: 2969, number of used features: 6140
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.525766 -> initscore=0.103156
[LightGBM] [Info] Start training from score 0.103156
word+char TF-IDF val AUC: 0.8812903993337828 (compare to char-only: 0.8730221675853591 )


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [59]:
iso_tfidf_v2 = IsotonicRegression(out_of_bounds='clip').fit(val_tfidf_probs_v2, y_val)
val_tfidf_cal_v2 = iso_tfidf_v2.predict(val_tfidf_probs_v2)
test_tfidf_cal_v2 = iso_tfidf_v2.predict(test_tfidf_probs_v2)

stack_X_val_v2 = np.column_stack([val_muril_cal, val_tfidf_cal_v2])
stack_X_test_v2 = np.column_stack([test_muril_cal, test_tfidf_cal_v2])
stacker_v2 = LR().fit(stack_X_val_v2, y_val)

prec, rec, thresh = precision_recall_curve(y_val, stacker_v2.predict_proba(stack_X_val_v2)[:,1])
f1_scores = [fbeta_score(y_val, stacker_v2.predict_proba(stack_X_val_v2)[:,1] >= t, beta=1) for t in thresh]
BEST_THRESHOLD_v2 = thresh[int(np.argmax(f1_scores))]

test_pred_v2 = (stacker_v2.predict_proba(stack_X_test_v2)[:,1] >= BEST_THRESHOLD_v2).astype(int)
print(classification_report(y_test, test_pred_v2, target_names=['NON-HATE','HATE']))

              precision    recall  f1-score   support

    NON-HATE       0.82      0.72      0.77       302
        HATE       0.77      0.86      0.81       335

    accuracy                           0.79       637
   macro avg       0.80      0.79      0.79       637
weighted avg       0.80      0.79      0.79       637



In [58]:
from collections import Counter
counts = Counter(y2_train)
max_count = max(counts.values())

train_emb_hate = train_emb[train_hate_mask]
oversample_idx = []
for cls, cnt in counts.items():
    cls_idx = np.where(y2_train == cls)[0]
    reps = max_count // cnt
    oversample_idx.extend(np.tile(cls_idx, reps))
oversample_idx = np.array(oversample_idx)

head2_balanced = LR(max_iter=1000, class_weight='balanced').fit(train_emb_hate[oversample_idx], y2_train[oversample_idx])
pred2_balanced = head2_balanced.predict(test_emb[test_hate_mask])

print('--- Level2, oversampled ---')
print(classification_report(y2_test, pred2_balanced, labels=np.unique(y2_test), target_names=encoders['Level2_Target'].classes_[np.unique(y2_test)], zero_division=0))
print('--- Level2, original (for comparison) ---')
print(classification_report(y2_test, pred2_test, labels=np.unique(y2_test), target_names=encoders['Level2_Target'].classes_[np.unique(y2_test)], zero_division=0))

--- Level2, oversampled ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.12      0.56      0.20        25
   Hate against Men       0.36      0.50      0.42        86
 Hate against Women       0.68      0.23      0.35       166
 No Target- Profane       0.16      0.11      0.13        47

           accuracy                           0.31       324
          macro avg       0.33      0.35      0.27       324
       weighted avg       0.48      0.31      0.32       324

--- Level2, original (for comparison) ---
                     precision    recall  f1-score   support

Hate against LGBTQ+       0.13      0.60      0.21        25
   Hate against Men       0.35      0.50      0.41        86
 Hate against Women       0.64      0.22      0.33       166
 No Target- Profane       0.20      0.11      0.14        47

           accuracy                           0.31       324
          macro avg       0.33      0.36      0.27       324
       wei

In [56]:
tfidf = tfidf_v2 if False else tfidf  # keep original tfidf variable name for predict() compatibility — see below
lgbm = lgbm_v2
stacker = stacker_v2
BEST_THRESHOLD = BEST_THRESHOLD_v2
iso_tfidf = iso_tfidf_v2

In [64]:
def predict(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
        cls_emb = model.backbone(**inputs).last_hidden_state[:, 0].cpu().numpy()
    muril_prob_raw = torch.softmax(out['logits1'], dim=-1)[0, HATE_IDX].item()
    muril_prob = iso_muril.predict([muril_prob_raw])[0]

    text_char = tfidf.transform([text])
    text_word = tfidf_word.transform([text])
    text_combined = hstack([text_char, text_word])
    tfidf_prob_raw = lgbm_v2.predict_proba(text_combined)[0, 1]
    tfidf_prob = iso_tfidf_v2.predict([tfidf_prob_raw])[0]

    stacked_prob = stacker_v2.predict_proba([[muril_prob, tfidf_prob]])[0, 1]
    label = 'HATE' if stacked_prob >= BEST_THRESHOLD_v2 else 'NON-HATE'
    result = {'Level1_Class': label, 'hate_prob': round(float(stacked_prob), 3), 'triage': triage(stacked_prob)}
    if label == 'HATE':
        result['Level2_Target'] = encoders['Level2_Target'].inverse_transform(head2.predict(cls_emb))[0]
        result['Level3_Category'] = encoders['Level3_Category'].inverse_transform(head3.predict(cls_emb))[0]
    else:
        result['Level2_Target'] = 'Nil'; result['Level3_Category'] = 'Nil'
    return result

In [65]:
from google.colab import files
import pickle, os

with open('final_artifacts_v2.pkl', 'wb') as f:
    pickle.dump({
        'tfidf_char': tfidf,
        'tfidf_word': tfidf_word,
        'lgbm': lgbm_v2,
        'stacker': stacker_v2,
        'threshold': BEST_THRESHOLD_v2,
        'head2': head2,
        'head3': head3,
        'encoders': encoders,
        'iso_muril': iso_muril,
        'iso_tfidf': iso_tfidf_v2
    }, f)

files.download('final_artifacts_v2.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>